In [ ]:
%pip install lightgbm catboost scikit-learn pandas numpy matplotlib kagglehub

In [ ]:
import kagglehub
import pandas as pd
import zipfile
import os
import numpy as np

print("="*80)
print("ECOGRID AI: REAL DATASET INTEGRATION PIPELINE")
print("="*80)

# Download first dataset (ASHRAE energy prediction)
print("\n[STEP 1/4] Downloading ASHRAE energy prediction dataset...")
path1 = kagglehub.competition_download('ashrae-energy-prediction')
print("Path to competition files:", path1)

# Load ASHRAE datasets
print("\n[STEP 2/4] Loading ASHRAE datasets...")
train_df = pd.read_csv(os.path.join(path1, 'train.csv'))
building_meta = pd.read_csv(os.path.join(path1, 'building_metadata.csv'))
weather_train = pd.read_csv(os.path.join(path1, 'weather_train.csv'))

print(f"  - Training data: {train_df.shape}")
print(f"  - Building metadata: {building_meta.shape}")
print(f"  - Weather data: {weather_train.shape}")

# Extract and load occupancy detection dataset
print("\n[STEP 3/4] Loading UCI Occupancy Detection dataset...")
second_dataset_path = "occupancy+detection.zip"
extract_dir = "occupancy_extracted"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(second_dataset_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

occupancy_train = pd.read_csv(os.path.join(extract_dir, 'datatraining.txt'))
occupancy_test1 = pd.read_csv(os.path.join(extract_dir, 'datatest.txt'))
occupancy_test2 = pd.read_csv(os.path.join(extract_dir, 'datatest2.txt'))

# Combine all occupancy data
occupancy_combined = pd.concat([occupancy_train, occupancy_test1, occupancy_test2], ignore_index=True)
print(f"  - Combined occupancy data: {occupancy_combined.shape}")

# =============================================================================
# ECOGRID FEATURE INTEGRATION MAPPING
# =============================================================================
print("\n[STEP 4/4] Creating Integrated EcoGrid Feature Matrix...")

# Convert timestamps to datetime and align to hourly granularity
train_df['timestamp'] = pd.to_datetime(train_df['timestamp'])
weather_train['timestamp'] = pd.to_datetime(weather_train['timestamp'])
occupancy_combined['date'] = pd.to_datetime(occupancy_combined['date'])

# Filter ASHRAE data for electricity meters (meter = 0) and chilled water (meter = 1)
electricity_df = train_df[train_df['meter'] == 0].copy()  # Electricity
chilled_water_df = train_df[train_df['meter'] == 1].copy()  # Chilled water

print(f"  - Electricity readings: {electricity_df.shape}")
print(f"  - Chilled water readings: {chilled_water_df.shape}")

# Merge ASHRAE data (training + building metadata + weather)
ashrae_merged = pd.merge(electricity_df, building_meta, on='building_id', how='left')
ashrae_merged = pd.merge(ashrae_merged, weather_train, on=['site_id', 'timestamp'], how='left')

print(f"  - ASHRAE merged shape: {ashrae_merged.shape}")

# Create hourly timestamp key for both datasets
ashrae_merged['hourly_timestamp'] = ashrae_merged['timestamp'].dt.floor('H')
occupancy_combined['hourly_timestamp'] = occupancy_combined['date'].dt.floor('H')

# Map UCI Occupancy features to EcoGrid naming convention
occupancy_features = occupancy_combined.groupby('hourly_timestamp').agg({
    'Temperature': 'mean',           # Ambient_Temp_C
    'Humidity': 'mean',             # Additional environmental feature
    'Light': 'mean',                # Additional environmental feature  
    'CO2': 'mean',                  # CO2 levels for occupancy inference
    'HumidityRatio': 'mean',        # Additional environmental feature
    'Occupancy': 'max'              # Occupancy_State (binary: 0 or 1)
}).reset_index()

# Rename columns to match EcoGrid feature names
occupancy_features = occupancy_features.rename(columns={
    'Temperature': 'Ambient_Temp_C_UCI',
    'CO2': 'CO2_Level',
    'Occupancy': 'Occupancy_State_UCI'
})

print(f"  - UCI occupancy features shape: {occupancy_features.shape}")

# Create EcoGrid feature matrix from ASHRAE data
ecogrid_features = ashrae_merged.copy()

# Map ASHRAE features to EcoGrid naming convention
ecogrid_features = ecogrid_features.rename(columns={
    'air_temperature': 'Ambient_Temp_C_ASHRAE',
    'meter_reading': 'HVAC_Power_kW_Raw'
})

# Create inferred occupancy from building characteristics
# High occupancy = high floor count + commercial primary use
ecogrid_features['Inferred_Occupancy'] = np.where(
    (ecogrid_features['primary_use'].isin(['Education', 'Office', 'Services'])) & 
    (ecogrid_features['floor_count'] > 3), 1, 0
)

# Create time-based features (Hour_Sin, Hour_Cos, DayOfWeek, IsWeekend)
ecogrid_features['Hour'] = ecogrid_features['hourly_timestamp'].dt.hour
ecogrid_features['DayOfWeek'] = ecogrid_features['hourly_timestamp'].dt.dayofweek
ecogrid_features['IsWeekend'] = (ecogrid_features['DayOfWeek'] >= 5).astype(int)

# Cyclical time features
ecogrid_features['Hour_Sin'] = np.sin(2 * np.pi * ecogrid_features['Hour'] / 24.0)
ecogrid_features['Hour_Cos'] = np.cos(2 * np.pi * ecogrid_features['Hour'] / 24.0)

# Temperature rolling mean (3-hour window)
ecogrid_features['Temp_Rolling_Mean'] = ecogrid_features['Ambient_Temp_C_ASHRAE'].rolling(
    window=3, min_periods=1).mean()

# =============================================================================
# DUAL-TRACK INTEGRATION: Join on hourly timestamp
# =============================================================================
print("\n" + "="*80)
print("DUAL-TRACK ENGINE: INTEGRATING SPATIAL + KINEMATIC FEATURES")
print("="*80)

# Perform the integration join
integrated_matrix = pd.merge(
    ecogrid_features, 
    occupancy_features, 
    on='hourly_timestamp', 
    how='left'
)

print(f"Integrated EcoGrid Feature Matrix shape: {integrated_matrix.shape}")
print(f"Available features: {integrated_matrix.columns.tolist()}")

# Create final EcoGrid feature columns with preference hierarchy
# Priority: Real sensor data > Inferred > Missing
integrated_matrix['Ambient_Temp_C'] = integrated_matrix['Ambient_Temp_C_UCI'].fillna(
    integrated_matrix['Ambient_Temp_C_ASHRAE']
)

integrated_matrix['Occupancy_State'] = integrated_matrix['Occupancy_State_UCI'].fillna(
    integrated_matrix['Inferred_Occupancy']
)

# Convert occupancy to categorical (High/Medium/Low) for EcoGrid
def map_occupancy_category(row):
    if pd.isna(row['Occupancy_State']):
        return 'Low'
    if row['CO2_Level'] > 1000:  # High CO2 indicates high occupancy
        return 'High'
    elif row['CO2_Level'] > 600:
        return 'Medium'
    else:
        return 'High' if row['Occupancy_State'] == 1 else 'Low'

integrated_matrix['Occupancy_Category'] = integrated_matrix.apply(map_occupancy_category, axis=1)

# Final feature selection for EcoGrid AI
final_features = integrated_matrix[[
    'hourly_timestamp',
    'Hour_Sin', 
    'Hour_Cos', 
    'DayOfWeek', 
    'IsWeekend', 
    'Ambient_Temp_C',
    'Temp_Rolling_Mean',
    'Occupancy_Category',
    'HVAC_Power_kW_Raw',
    'CO2_Level',
    'Light',
    'Humidity'
]].copy()

# Rename for EcoGrid compatibility
final_features = final_features.rename(columns={
    'HVAC_Power_kW_Raw': 'HVAC_Power_kW'
})

print(f"\nFinal EcoGrid Feature Matrix shape: {final_features.shape}")
print(f"Final features: {final_features.columns.tolist()}")

# Save the integrated dataset
final_features.to_csv('ecogrid_integrated_matrix.csv', index=False)
print("\n" + "="*80)
print("✅ INTEGRATED ECOGRID FEATURE MATRIX SAVED: ecogrid_integrated_matrix.csv")
print("="*80)

# Display sample of integrated data
print("\nSample of integrated data:")
print(final_features.head(10))

In [ ]:
# =====================================================================
# ECOGRID AI: MULTI-TASK HETEROGENEOUS GRADIENT BOOSTING ENSEMBLE
# Production Build | Real Dataset Integration
# =====================================================================

import sys
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend to force rendering image files
import matplotlib.pyplot as plt

import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
np.random.seed(42)

def console_log(msg: str, delay: float = 0.01):
    """Outputs structured log messages directly to standard output."""
    print(msg, flush=True)
    time.sleep(delay)

class EcoGridEngine:
    def __init__(self):
        self.label_encoder = LabelEncoder()
        self.feature_cols = ['Hour_Sin', 'Hour_Cos', 'DayOfWeek', 'IsWeekend', 'Ambient_Temp_C', 'Temp_Rolling_Mean']

        # Hyperparameters with L2 Regularization & Tree Depth Constraints
        self.lgb_params = {'n_estimators': 100, 'max_depth': 4, 'learning_rate': 0.03, 'reg_lambda': 8.0, 'random_state': 42, 'verbose': -1}
        self.cat_params = {'iterations': 100, 'depth': 4, 'learning_rate': 0.03, 'l2_leaf_reg': 8.0, 'random_state': 42, 'verbose': 0}

    def run_pipeline(self):
        console_log("="*80)
        console_log(" 🚀 ECOGRID AI: MULTI-TASK GRADIENT BOOSTING PIPELINE (REAL DATA)")
        console_log("="*80)

        # 1. Load Real Integrated Dataset
        console_log("\n[STEP 1/4] Loading Real Integrated EcoGrid Dataset...")
        
        try:
            df = pd.read_csv('ecogrid_integrated_matrix.csv')
            console_log(f" ↳ Loaded integrated dataset: {df.shape}")
            console_log(f" ↳ Available columns: {df.columns.tolist()}")
        except FileNotFoundError:
            console_log(" ❌ Error: ecogrid_integrated_matrix.csv not found. Run the data integration cell first.")
            return

        # 2. Data Preprocessing
        console_log("\n[STEP 2/4] Preprocessing Real Data...")
        
        # Handle missing values
        df = df.dropna(subset=['HVAC_Power_kW', 'Occupancy_Category'])
        console_log(f" ↳ After dropping missing values: {df.shape}")
        
        # Encode occupancy categories
        df['Occupancy_Label'] = self.label_encoder.fit_transform(df['Occupancy_Category'])
        console_log(f" ↳ Occupancy classes: {self.label_encoder.classes_}")
        
        # Ensure features exist
        for col in self.feature_cols:
            if col not in df.columns:
                console_log(f" ❌ Error: Required feature '{col}' not found in dataset")
                return

        # 3. Strict Sequential Partitioning (No Time-Series Leakage)
        X = df[self.feature_cols]
        y_cls = df["Occupancy_Label"]
        y_reg = df["HVAC_Power_kW"]

        X_train, X_test, y_train_cls, y_test_cls = train_test_split(X, y_cls, test_size=0.2, shuffle=False)
        _, _, y_train_reg, y_test_reg = train_test_split(X, y_reg, test_size=0.2, shuffle=False)

        console_log(f" ↳ Dataset: {len(df)} Rows | Train Window: {len(X_train)} Hours | Evaluation Horizon: {len(X_test)} Hours")

        # 4. Model Training
        console_log("\n[STEP 3/4] Fitting Dual-Track Heterogeneous Ensemble...")
        m1_cls = lgb.LGBMClassifier(**self.lgb_params).fit(X_train, y_train_cls)
        m2_cls = CatBoostClassifier(**self.cat_params).fit(X_train, y_train_cls)

        m1_reg = lgb.LGBMRegressor(**self.lgb_params).fit(X_train, y_train_reg)
        m2_reg = CatBoostRegressor(**self.cat_params).fit(X_train, y_train_reg)

        # 5. Metric Extraction
        console_log("\n[STEP 4/4] Evaluating Model Generalization Metrics...")
        tr_prob = (m1_cls.predict_proba(X_train) + m2_cls.predict_proba(X_train)) / 2
        te_prob = (m1_cls.predict_proba(X_test) + m2_cls.predict_proba(X_test)) / 2
        train_acc = accuracy_score(y_train_cls, np.argmax(tr_prob, axis=1))
        test_acc = accuracy_score(y_test_cls, np.argmax(te_prob, axis=1))

        te_pred_reg = (m1_reg.predict(X_test) + m2_reg.predict(X_test)) / 2
        tr_pred_reg = (m1_reg.predict(X_train) + m2_reg.predict(X_train)) / 2
        train_rmse = np.sqrt(mean_squared_error(y_train_reg, tr_pred_reg))
        test_rmse = np.sqrt(mean_squared_error(y_test_reg, te_pred_reg))

        console_log(f" 📑 Classification Accuracy -> Train: {train_acc*100:.1f}% | Test: {test_acc*100:.1f}%")
        console_log(f" 📑 Forecasting RMSE       -> Train: {train_rmse:.2f} kW | Test: {test_rmse:.2f} kW")

        # Live Endpoint Test with real data characteristics
        self.live_api_endpoint(m1_cls, m2_cls, m1_reg, m2_reg, hour=14, day_of_week=2, temp=34.5, temp_avg=33.8)

        # Generate Chart Output File
        self.plot_and_save(df, y_test_reg, te_pred_reg)

    def live_api_endpoint(self, m1_cls, m2_cls, m1_reg, m2_reg, hour, day_of_week, temp, temp_avg):
        console_log("\n[STEP 5/5] Executing Live Production API Emulation...")
        hour_sin = np.sin(2 * np.pi * hour / 24.0)
        hour_cos = np.cos(2 * np.pi * hour / 24.0)
        is_weekend = 1 if day_of_week >= 5 else 0

        payload = pd.DataFrame([[hour_sin, hour_cos, day_of_week, is_weekend, temp, temp_avg]], columns=self.feature_cols)

        voted_probs = (m1_cls.predict_proba(payload) + m2_cls.predict_proba(payload)) / 2
        assigned_state = self.label_encoder.classes_[np.argmax(voted_probs, axis=1)[0]]

        blended_pred = (m1_reg.predict(payload)[0] + m2_reg.predict(payload)[0]) / 2

        console_log(f" ↳ Incoming Telemetry --> {hour}:00 | DayOfWeek: {day_of_week} | Temp: {temp}°C")
        console_log(f" ↳ Spatial Model State --> [{assigned_state.upper()}] Occupancy")
        console_log(f" ↳ Kinetic Forecast    --> [{blended_pred:.2f} kW] Electrical Load")

        if assigned_state == "High" and blended_pred > 40.0:
            console_log(" 🚨 ROUTING ACTION    --> PRE-COOLING ENGAGED. Buffering peak spike.")
        elif assigned_state == "Low":
            console_log(" 🍃 ROUTING ACTION    --> DEEP HIBERNATE MODE ENGAGED.")
        else:
            console_log(" ⚖️ ROUTING ACTION    --> STEADY STATE MAINTAINED.")
        console_log("="*80)

    def plot_and_save(self, df, y_test_reg, te_pred_reg):
        plt.figure(figsize=(12, 5))
        plt.plot(df["hourly_timestamp"].iloc[-len(y_test_reg):].values, y_test_reg.values, label="Actual Load (kW)", color="steelblue", linewidth=2)
        plt.plot(df["hourly_timestamp"].iloc[-len(y_test_reg):].values, te_pred_reg, label="Predicted Load (kW)", color="darkorange", linewidth=2, linestyle="--")
        plt.title("EcoGrid AI: Real Data - Actual vs Predicted HVAC Power Consumption", fontsize=14, fontweight="bold")
        plt.xlabel("Timestamp", fontsize=12)
        plt.ylabel("Power (kW)", fontsize=12)
        plt.legend(fontsize=11)
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig("ecogrid_real_data_predictions.png", dpi=150)
        console_log(" 📊 Prediction plot saved: ecogrid_real_data_predictions.png")

# Run the EcoGrid AI pipeline with real data
if __name__ == "__main__":
    engine = EcoGridEngine()
    engine.run_pipeline()